# 02 — Relabel TikTok Data: Behavioral vs Full

**Input**: `gpt_label_processed_amplifiers.csv` (Lindsey's rule-labeled TikTok data)

**Output**: `tiktok_labeled.csv` with two labels per post:
- `amp_prob_full` — Lindsey's original label (unchanged)
- `amp_prob_behavioral` — original label, but positives surviving only if their trigger reason is *language-agnostic*

## Why two labels?

Lindsey's `analyze_comment()` mixes two kinds of rules:

1. **Behavioral / language-agnostic** — emoji density, length anomaly, emoji-to-text ratio, posts-with-no-text. These trigger on signals that exist on any platform in any language.
2. **Filipino-lexicon** — `lugaw`, `pinklawan`, `grabe`, `best president`, location names. These can only fire on Filipino political content.

**56% of the original positive class is defined by Filipino-lexicon rules.** A classifier trained on `amp_prob_full` and tested on Pantip will appear to fail at "cross-platform transfer" when really it's failing at "speaking Thai." That's a confound, not a finding.

By producing `amp_prob_behavioral` (positives triggered by language-agnostic rules only), we get a label whose definition does not depend on the input language. Cross-platform transfer on this label is a meaningful test.

## The thesis story

- **Headline experiment**: train on `amp_prob_behavioral`, test on Pantip → does behavioral amplification signal generalize?
- **Ablation experiment**: train on `amp_prob_full`, test on Pantip → confirms the lexicon-dependent portion collapses

The gap between the two F1 scores is the contribution: *quantifying how much of an amplifier-detection model's apparent performance is platform-bound vs platform-universal*.

In [1]:
import pandas as pd

IN_PATH  = 'Lindsey_dataset/gpt_label_processed_amplifiers.csv'
OUT_PATH = 'tiktok_labeled.csv'

df = pd.read_csv(IN_PATH)
df['amp_prob_full'] = df['amp_prob']
print(f'Loaded {len(df):,} posts')
print()
print('Original label distribution:')
print(df['amp_prob_full'].value_counts().sort_index(ascending=False))
print()
print(f'Positives (>=0.5): {(df["amp_prob_full"]>=0.5).sum():,} ({(df["amp_prob_full"]>=0.5).mean():.1%})')

Loaded 19,842 posts

Original label distribution:
amp_prob_full
1.00    1205
0.90     558
0.80    1525
0.70      21
0.67    1189
0.30     415
0.20    7151
0.15    3004
0.10    4774
Name: count, dtype: int64

Positives (>=0.5): 4,498 (22.7%)


## 1. Classify each trigger as Behavioral or Lexicon

Reading `analyze_comment()` in `labeling.ipynb`, each reason category is mapped to one bucket. The mapping is decided ONCE here, then applied deterministically.

**Behavioral (language-agnostic)** — fires on text structure, emoji count, character length. No Filipino phrases involved:
- `Excessive emojis (N) - algorithmic trigger`
- `High emoji count (N) minimal text`
- `High emoji-to-text ratio (N emojis, M words)`
- `Only emojis (N) no text`  *(these are negatives at prob=0.3, irrelevant for positive class but included for completeness)*
- `Substantial text - appears genuine` *(negative)*
- `Moderate text length - likely genuine` *(negative)*
- `Neutral - no clear amplification indicators` *(negative)*

**Lexicon (Filipino-specific)** — fires only when a Filipino political word/phrase appears:
- `Political attack term: "..."` (lugaw, pinklawan, meryenda, rappler)
- `BBM account tagging - coordination`
- `Indirect attack disguised as fact`
- `Algorithmic gaming: "..."` (fyp, viral, trending — *English/Filipino mixed; on Pantip equivalent terms don't exist*)
- `Cross-platform coordination: "..."` (facebook, fb page, twitter, etc.)
- `Single political name + emojis only`
- `Low-effort political phrase`
- `Emotional manipulation: "..."` (grabe, goosebumps, nakakaiyak)
- `Manufactured authenticity: "..."` (first time voter ako)
- `Location targeting: "..."` (manila, cebu, davao)
- `Emotional appeal: "..."` (deserve, proud of you)
- `Overuse of superlatives: "..."` (best president ever)
- `Coordination phrase: "..."` (tara na mga, mga solid, team bbm)

**Note on `Algorithmic gaming`**: ambiguous case. Strictly the words `fyp`, `viral`, `trending` are English and could in principle appear in Thai posts. But they're TikTok-platform-specific vocabulary that doesn't fit forum culture. I'm classifying them as lexicon (won't generalize to Pantip). If you want to be more permissive, you can move this one to Behavioral and rerun.

In [2]:
# Behavioral reason prefixes — match by startswith()
BEHAVIORAL_PREFIXES = (
    'Excessive emojis',
    'High emoji count',
    'High emoji-to-text ratio',
    'Only emojis',
    'Substantial text',
    'Moderate text length',
    'Neutral - no clear',
)

def classify_reason(reason):
    if not isinstance(reason, str):
        return 'unknown'
    return 'behavioral' if reason.startswith(BEHAVIORAL_PREFIXES) else 'lexicon'

df['trigger_type'] = df['reason'].apply(classify_reason)

print('Trigger type breakdown:')
print(df['trigger_type'].value_counts())
print()
print('Trigger type × positive label crosstab:')
ct = pd.crosstab(df['trigger_type'], df['amp_prob_full']>=0.5, margins=True)
ct.columns = ['negative', 'positive', 'total']
print(ct)
print()
# Sanity check: behavioral should have no entries in the Filipino-only categories
print('Sample behavioral reasons (should be emoji/length only):')
print(df[df['trigger_type']=='behavioral']['reason'].value_counts().head(8))
print()
print('Sample lexicon reasons (should be Filipino-phrase triggers):')
print(df[df['trigger_type']=='lexicon']['reason'].value_counts().head(8))

Trigger type breakdown:
trigger_type
behavioral    17276
lexicon        2566
Name: count, dtype: int64

Trigger type × positive label crosstab:
              negative  positive  total
trigger_type                           
behavioral       15304      1972  17276
lexicon             40      2526   2566
All              15344      4498  19842

Sample behavioral reasons (should be emoji/length only):
reason
Neutral - no clear amplification indicators     7151
Substantial text - appears genuine              4774
Moderate text length - likely genuine           3004
High emoji-to-text ratio (4 emojis, 1 words)     194
Only emojis (3) no text                          186
High emoji-to-text ratio (6 emojis, 1 words)     135
High emoji-to-text ratio (4 emojis, 0 words)     124
Excessive emojis (10) - algorithmic trigger      122
Name: count, dtype: int64

Sample lexicon reasons (should be Filipino-phrase triggers):
reason
Emotional manipulation: "grabe"         338
Low-effort political phrase 

## 2. Build `amp_prob_behavioral`

Strategy:
- For **positives** (`amp_prob_full >= 0.5`): keep value if `trigger_type == 'behavioral'`, else drop to `0.2` (the "neutral" default in Lindsey's pipeline).
- For **negatives** (`amp_prob_full < 0.5`): keep value unchanged — negatives are still negatives.

This means a post originally labeled `amp_prob=1.0 / reason="Political attack term: lugaw"` becomes `amp_prob_behavioral=0.2` (no longer a positive). A post labeled `amp_prob=1.0 / reason="Excessive emojis (15)"` keeps `amp_prob_behavioral=1.0`.

In [3]:
df['amp_prob_behavioral'] = df['amp_prob_full'].copy()

mask_drop = (df['amp_prob_full'] >= 0.5) & (df['trigger_type'] == 'lexicon')
df.loc[mask_drop, 'amp_prob_behavioral'] = 0.2

print(f'Posts demoted from positive → 0.2: {mask_drop.sum():,}')
print()
print('amp_prob_behavioral distribution:')
print(df['amp_prob_behavioral'].value_counts().sort_index(ascending=False))
print()
print(f'Behavioral positives (>=0.5): {(df["amp_prob_behavioral"]>=0.5).sum():,} '
      f'({(df["amp_prob_behavioral"]>=0.5).mean():.1%})')
print(f'Full positives (>=0.5):       {(df["amp_prob_full"]>=0.5).sum():,} '
      f'({(df["amp_prob_full"]>=0.5).mean():.1%})')

Posts demoted from positive → 0.2: 2,526

amp_prob_behavioral distribution:
amp_prob_behavioral
1.00     783
0.67    1189
0.30     415
0.20    9677
0.15    3004
0.10    4774
Name: count, dtype: int64

Behavioral positives (>=0.5): 1,972 (9.9%)
Full positives (>=0.5):       4,498 (22.7%)


## 3. Confirm the behavioral positives are language-agnostic

Every surviving positive should be triggered by emoji/length structure, not by a Filipino phrase.

In [4]:
behav_pos = df[df['amp_prob_behavioral'] >= 0.5]
print(f'Behavioral positive count: {len(behav_pos):,}')
print()
print('Reason breakdown of behavioral positives:')
print(behav_pos['reason'].value_counts().head(15))
print()
# Look at a few examples
print('Example behavioral positives:')
for i, row in behav_pos.sample(min(5, len(behav_pos)), random_state=42).iterrows():
    text = row['text'][:80] + '...' if len(str(row['text'])) > 80 else row['text']
    print(f'  [{row["amp_prob_behavioral"]:.2f}] "{text}"')
    print(f'         reason: {row["reason"]}')

Behavioral positive count: 1,972

Reason breakdown of behavioral positives:
reason
High emoji-to-text ratio (4 emojis, 1 words)    194
High emoji-to-text ratio (6 emojis, 1 words)    135
High emoji-to-text ratio (4 emojis, 0 words)    124
Excessive emojis (10) - algorithmic trigger     122
High emoji-to-text ratio (4 emojis, 3 words)     95
High emoji-to-text ratio (5 emojis, 1 words)     93
High emoji-to-text ratio (4 emojis, 4 words)     89
Excessive emojis (12) - algorithmic trigger      82
High emoji count (8) minimal text                82
High emoji count (7) minimal text                81
Excessive emojis (11) - algorithmic trigger      72
High emoji-to-text ratio (5 emojis, 0 words)     71
High emoji-to-text ratio (4 emojis, 2 words)     70
High emoji-to-text ratio (6 emojis, 0 words)     61
High emoji-to-text ratio (6 emojis, 4 words)     52
Name: count, dtype: int64

Example behavioral positives:
  [1.00] "🥰🥰🥰🥰🥰🥰🥰✌️✌️✌️✌️✌️✌️✌️"
         reason: Excessive emojis (14) - algori

## 4. Export

In [5]:
out_cols = ['cid', 'text', 'amp_prob_full', 'amp_prob_behavioral', 'trigger_type', 'reason']
df[out_cols].to_csv(OUT_PATH, index=False)
print(f'Saved → {OUT_PATH}')
print(f'  Rows: {len(df):,}')
print(f'  Columns: {out_cols}')
print()
print('Summary:')
print(f'  Total posts:           {len(df):,}')
print(f'  Full positives:        {(df["amp_prob_full"]>=0.5).sum():,} ({(df["amp_prob_full"]>=0.5).mean():.1%})')
print(f'  Behavioral positives:  {(df["amp_prob_behavioral"]>=0.5).sum():,} ({(df["amp_prob_behavioral"]>=0.5).mean():.1%})')
print(f'  Demoted (lex-only):    {mask_drop.sum():,}')
print()
print('Next: re-run 00_merged_preprocessing.ipynb pointing TikTok side at tiktok_labeled.csv')
print('      then re-run 03_common_features.ipynb at POST level (no user aggregation)')

Saved → tiktok_labeled.csv
  Rows: 19,842
  Columns: ['cid', 'text', 'amp_prob_full', 'amp_prob_behavioral', 'trigger_type', 'reason']

Summary:
  Total posts:           19,842
  Full positives:        4,498 (22.7%)
  Behavioral positives:  1,972 (9.9%)
  Demoted (lex-only):    2,526

Next: re-run 00_merged_preprocessing.ipynb pointing TikTok side at tiktok_labeled.csv
      then re-run 03_common_features.ipynb at POST level (no user aggregation)
